In [ ]:
import numpy as np
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
import re
from nltk.stem import WordNetLemmatizer
import tqdm
import random

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, LSTM, Concatenate, Dense, Input,Dropout
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers.legacy import Adam
from sklearn.model_selection import train_test_split

from transformers import BertTokenizer, BertForSequenceClassification, RobertaTokenizer, RobertaForSequenceClassification, AdamW

In [ ]:
# Specify the folder path where your CSV files are located
folder_path = "/Users/kianathakkar/Documents/dataset/quora-question-pairs/train.csv"

df_train = pd.read_csv(folder_path)

In [ ]:
# Specify the folder path where your CSV files are located
folder_path_test = "/Users/kianathakkar/Documents/dataset/quora-question-pairs/test.csv"

df_test = pd.read_csv(folder_path_test)

In [ ]:
df_train.head()

In [ ]:
df_train = df_train.dropna()

In [ ]:
# Tokenization and Lemmatization
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

In [ ]:
# Preprocessing and Cleaning Text Data
def preprocess_text(text):
    
    # Check if the input is a non-empty string
    if isinstance(text, str) and text.strip():  # Check if it's a non-empty string
        # Lowercase the text
        text = text.lower()

        # Replace certain special characters with their string equivalents
        text = text.replace('%', ' percent')
        text = text.replace('$', ' dollar ')
        text = text.replace('?', ' rupee ')
        text = text.replace('?', ' euro ')
        text = text.replace('@', ' at ')
        
        # The pattern '[math]' appears around 900 times in the whole dataset.
        text = text.replace('[math]', '')       

        # Remove special characters and numbers
        text = re.sub(r'[^a-zA-Z\s]', '', text)
         
    return text

In [ ]:
# Define a dictionary of common English contractions and their expanded forms
contractions_dict = {
    "ain't": "is not",
    "aren't": "are not",
    "can't": "cannot",
    "could've": "could have",
    "couldn't": "could not",
    "didn't": "did not",
    "doesn't": "does not",
    "don't": "do not",
    "hadn't": "had not",
    "hasn't": "has not",
    "haven't": "have not",
    "he'd": "he would",
    "he'll": "he will",
    "he's": "he is",
    "I'd": "I would",
    "I'll": "I will",
    "I'm": "I am",
    "I've": "I have",
    "isn't": "is not",
    "it's": "it is",
    "let's": "let us",
    "might've": "might have",
    "mustn't": "must not",
    "shan't": "shall not",
    "she'd": "she would",
    "she'll": "she will",
    "she's": "she is",
    "should've": "should have",
    "shouldn't": "should not",
    "that's": "that is",
    "there's": "there is",
    "they'd": "they would",
    "they'll": "they will",
    "they're": "they are",
    "they've": "they have",
    "we'd": "we would",
    "we're": "we are",
    "we've": "we have",
    "weren't": "were not",
    "what's": "what is",
    "when's": "when is",
    "where's": "where is",
    "who's": "who is",
    "why's": "why is",
    "won't": "will not",
    "would've": "would have",
    "wouldn't": "would not",
    "you'd": "you would",
    "you'll": "you will",
    "you're": "you are",
    "you've": "you have"
}

# Function to expand contractions
def expand_contractions(text, contractions_dict):
    
    if isinstance(text, str):
        words = text.split()
        expanded_words = [contractions_dict.get(word, word) for word in words]
        expanded_text = " ".join(expanded_words)
        return expanded_text
    else:
        return text 
    

In [ ]:
# Tokenization and Lemmatization
def tokenize_and_lemmatize(text):
    if isinstance(text, str):
        words = text.split()
        words = [lemmatizer.lemmatize(word) for word in words]
        return ' '.join(words)
    else:
        return text  # Return the input as is if it's not a string

In [ ]:

# List of column names to clean
columns_to_clean = ['question1','question2']

# Apply the functions to each column in the DataFrame
for column in columns_to_clean:
    # Apply the special character removal function
    df_train[column] = df_train[column].apply(preprocess_text)
    df_test[column] = df_test[column].apply(preprocess_text)

    # Apply the contraction expansion function
    df_train[column] = df_train[column].apply(expand_contractions, args=(contractions_dict,))
    df_test[column] = df_test[column].apply(expand_contractions, args=(contractions_dict,))   
     
    # Apply the stop word removal function
    df_train[column] = df_train[column].apply(tokenize_and_lemmatize)
    df_test[column] = df_test[column].apply(tokenize_and_lemmatize)

In [ ]:
df_train['question1'] = df_train['question1'].astype(str)
df_train['question2'] = df_train['question2'].astype(str)

In [ ]:
# Assuming you have preprocessed text in 'question1' and 'question2' columns
question1_texts = df_train['question1'].apply(lambda x: str(x))
question2_texts = df_train['question2'].apply(lambda x: str(x))

# Tokenize the text in both columns
question1_tokens = [word_tokenize(text) for text in question1_texts]
question2_tokens = [word_tokenize(text) for text in question2_texts]

In [ ]:
import matplotlib.pyplot as plt

# Create a list to store the lengths of the internal lists
lengths = [len(internal_list) for internal_list in question1_tokens]

# Create a histogram
plt.hist(lengths, bins=20)  # Adjust the number of bins as needed
plt.xlabel('Length of Question')
plt.ylabel('Frequency')
plt.title('Histogram of Question Lengths')

# Show the histogram
plt.show()

In [ ]:
# Tokenization and sequence padding
max_seq_length = 40  # Define your maximum sequence length
tokenizer = Tokenizer()
tokenizer.fit_on_texts(question1_tokens + question2_tokens)

question1_sequences = tokenizer.texts_to_sequences(question1_tokens)
question2_sequences = tokenizer.texts_to_sequences(question2_tokens)
question1_sequences = pad_sequences(question1_sequences, maxlen=max_seq_length)
question2_sequences = pad_sequences(question2_sequences, maxlen=max_seq_length)


In [ ]:
# Split data into training and testing sets
X = np.column_stack((question1_sequences, question2_sequences))
y = df_train['is_duplicate'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Build the LSTM model
embedding_dim = 100  
lstm_units = 64

input1 = Input(shape=(max_seq_length,))
input2 = Input(shape=(max_seq_length,))

embedding_layer = Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=embedding_dim)
lstm_layer = LSTM(lstm_units)

encoded1 = lstm_layer(embedding_layer(input1))
encoded2 = lstm_layer(embedding_layer(input2))

In [ ]:
embedding_weights = embedding_layer.get_weights()[0]
print(embedding_weights.shape)

In [ ]:
# Concatenate 
merged = Concatenate()([encoded1, encoded2])

# Add one or more Dense layers
dense_units = 128  # Define the number of units in the dense layers

dense1 = Dense(dense_units, activation='relu', kernel_initializer=HeNormal())(merged)  # Add a dense layer after the Concatenate layer
dense2 = Dense(dense_units, activation='relu', kernel_initializer=HeNormal())(dense1)  # Optionally, you can add more dense layers

# Add the final output layer
output = Dense(1, activation='sigmoid')(dense2)

model_lstm = Model(inputs=[input1, input2], outputs=output)

model_lstm.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])


In [ ]:
model_lstm.summary()

In [ ]:
from keras.utils.vis_utils import plot_model

plot_model(model_lstm, to_file='model_plot.png', show_shapes=True, show_layer_names=True)

In [ ]:
# Train the model
model_lstm.fit([X_train[:, :max_seq_length], X_train[:, max_seq_length:]], y_train, batch_size=64, epochs=5, validation_data=([X_test[:, :max_seq_length], X_test[:, max_seq_length:]], y_test))

### Getting the best threshold for the model

In [ ]:
from sklearn.metrics import precision_recall_curve,auc,f1_score

# Assuming you have a trained model 'model'
y_prob = model_lstm.predict([X_test[:, :max_seq_length], X_test[:, max_seq_length:]])

In [ ]:
best_threshold = 0
best_f1 = 0

for threshold in range(1, 100):
    threshold /= 100
    y_pred_binary = (y_prob > threshold).astype(int)
    f1 = f1_score(y_test, y_pred_binary)
    
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(f"Best Threshold: {best_threshold:.2f}")
print(f"Best F1 Score: {best_f1:.2f}")

### Precision Recall Curve

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)
area_under_curve = auc(recalls, precisions)  # Calculate the AUC

# Plot the precision-recall curve
plt.figure(figsize=(8, 6))
plt.plot(recalls, precisions, marker='.', label=f'PR AUC = {area_under_curve:.2f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.grid()
plt.show()

### Using the classifier with optimum threshold to get classification report

In [ ]:
from sklearn.metrics import classification_report

# Assuming you have a trained model 'model'
y_pred = model_lstm.predict([X_test[:, :max_seq_length], X_test[:, max_seq_length:]])

# Convert the predictions to binary labels (0 or 1)
y_pred_binary = (y_pred > 0.48).astype(int)

# Print the classification report
print(classification_report(y_test, y_pred_binary))

### Using Transformer Architecture for classification

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import BertTokenizer,BertForSequenceClassification
from transformers import DistilBertTokenizerFast,DistilBertForSequenceClassification
import torch

In [ ]:
# Create a DistilBERT tokenizer
tokenizer_dbert = BertTokenizer.from_pretrained('bert-base-uncased')

# Create a DistilBERT model
model_dbert = BertForSequenceClassification.from_pretrained('bert-base-uncased')

In [ ]:
random_indices = random.sample(range(len(df_train)), 5000)
df_sampled = df_train.iloc[random_indices]

#Sample 24,000 data points for training
training_indices = random.sample(range(len(df_sampled)), 4000)
df_train_v2 = df_sampled.iloc[training_indices]

# Create an evaluation DataFrame with the remaining 6,000 data points
evaluation_indices = [i for i in range(len(df_sampled)) if i not in training_indices]
df_eval = df_sampled.iloc[evaluation_indices]

In [ ]:
# Prepare sentence pairs and labels for training data
sentence_pairs_train = [(row['question1'], row['question2']) for _, row in df_train_v2.iterrows()]
labels_train = df_train_v2['is_duplicate'].tolist()

In [ ]:
#Prepare sentence pairs and labels for validation
sentence_pairs_eval = [(row['question1'], row['question2']) for _, row in df_eval.iterrows()]
labels_eval = df_eval['is_duplicate'].tolist()

In [ ]:
encodings_train = tokenizer_dbert(sentence_pairs_train,padding=True, truncation=True)

encodings_eval = tokenizer_dbert(sentence_pairs_eval, padding=True, truncation=True)

In [ ]:
type(encodings_train)

In [ ]:
first_record_input_ids = encodings_train['input_ids'][0]
first_record_attention_mask = encodings_train['attention_mask'][0]

In [ ]:
class QuestionPairClassificationDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

dataset_train = QuestionPairClassificationDataset(encodings_train, labels_train)
dataloader = torch.utils.data.DataLoader(dataset_train, batch_size=8, shuffle=True)

dataset_eval = QuestionPairClassificationDataset(encodings_eval, labels_eval)
dataloader_eval = torch.utils.data.DataLoader(dataset_eval, batch_size=8, shuffle=True)

In [ ]:
from sklearn.metrics import precision_score, recall_score, accuracy_score

def compute_custom_metrics(eval_prediction):
    predicted_labels = eval_prediction.predictions.argmax(axis=1)
    true_labels = eval_prediction.label_ids

    precision = precision_score(true_labels, predicted_labels)
    recall = recall_score(true_labels, predicted_labels)
    accuracy = accuracy_score(true_labels, predicted_labels)

    return {'precision': precision, 'recall': recall, 'accuracy': accuracy}

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    report_to="tensorboard",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    save_steps=400,
    save_total_limit=3,
    evaluation_strategy="steps",
    eval_steps=200,
)

In [ ]:
trainer = Trainer(
    model=model_dbert,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_eval,
    compute_metrics=compute_custom_metrics
)

In [ ]:
trainer.train()